# Modern Pipeline - Complete Workflow

This notebook demonstrates the **modern approach** to data processing using the complete toolkit.

**One command instead of manual processing for each dataset!**

## Setup

In [ ]:
from pathlib import Path
import sys

# Setup paths
REPO_ROOT = Path.cwd().resolve().parent
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"✅ Repository root: {REPO_ROOT}")

## 🚀 Part 1: Modern Pipeline (ONE COMMAND!)

Process all 5 datasets with a single function call:

In [ ]:
from clean import process_all_datasets, merge_all_datasets

# Process ALL datasets with one command!
results = process_all_datasets(
    repo_root=REPO_ROOT,
    year_min=1980,
    year_max=2023,
    validate=True,
    save_outputs=True
)

print(f"\n✅ Processed {len(results)} datasets!")
for name, df in results.items():
    if df is not None:
        print(f"   {name}: {len(df):,} observations")

In [ ]:
# Merge into master dataset
master = merge_all_datasets(results, how='outer')

print(f"\n📊 Master Dataset:")
print(f"   {len(master):,} observations")
print(f"   {master['iso3'].nunique()} countries")
print(f"   {master['year'].min()}-{master['year'].max()}")
print(f"   {len(master.columns)} variables")
master.head()

## 📈 Part 2: Data Quality Report

In [ ]:
from clean import generate_quality_report

# Generate comprehensive quality report
quality_report = generate_quality_report(
    master,
    output_path=REPO_ROOT / "reports" / "quality_report.html"
)

## 📊 Part 3: Summary Statistics

In [ ]:
from clean import generate_summary_stats

# Generate summary statistics
stats = generate_summary_stats(master)
stats

In [ ]:
# Export to LaTeX (for paper!)
latex_stats = generate_summary_stats(master, output_format='latex')
print("LaTeX Table (copy to paper):")
print(latex_stats)

## 🔬 Part 4: Statistical Tests

### Test Stationarity (Important for time series!)

In [ ]:
from clean import test_stationarity

# Test which variables are stationary
stationarity_results = test_stationarity(
    master,
    variables=['ln_gdppc', 'sstran', 'deficit', 'inflation_cpi'],
    test='adf'  # Augmented Dickey-Fuller test
)

stationarity_results

## ⚙️ Part 5: Prepare for Analysis

### Create Lags for Dynamic Models

In [ ]:
from clean import create_lags, check_panel_balance

# Check if panel is balanced
balance = check_panel_balance(master)
print(f"Panel balanced: {balance['balanced']}")

# Create lags for regression
master_with_lags = create_lags(
    master,
    variables=['ln_gdppc', 'deficit'],
    lags=[1, 2, 3]
)

print(f"\n✅ Added lag variables:")
lag_cols = [c for c in master_with_lags.columns if '_lag' in c]
print(lag_cols)

## 📊 Part 6: Visualization

In [ ]:
from clean import plot_time_series, plot_correlation_matrix
import matplotlib.pyplot as plt

# Plot GDP over time for selected countries
plot_time_series(
    master,
    'ln_gdppc',
    countries=['USA', 'DEU', 'GBR', 'FRA', 'SWE', 'DNK']
)

In [ ]:
# Correlation heatmap
plot_correlation_matrix(
    master,
    variables=['ln_gdppc', 'sstran', 'deficit', 'debt', 'inflation_cpi']
)

## 🔧 Part 7: Filter by Country Groups

In [ ]:
from clean import filter_by_region, list_regions

# See available regions
list_regions()

In [ ]:
# Filter to Nordic countries
nordic = filter_by_region(master, 'nordic')
print(f"Nordic countries: {nordic['iso3'].unique()}")
print(f"{len(nordic):,} observations")

## 🎯 Part 8: Robustness Checks

In [ ]:
from clean.robustness import run_robustness_checks, compare_robustness_results
from statsmodels.formula.api import ols

# Run model with automated robustness checks
robust_results = run_robustness_checks(
    master_with_lags,
    'sstran ~ ln_gdppc + deficit + ln_gdppc_lag1',
    ols,
    checks=['drop_outliers', 'winsorize', 'pre_2008', 'post_2008']
)

In [ ]:
# Compare coefficients across specifications
comparison = compare_robustness_results(robust_results, variable='ln_gdppc')

## 📄 Part 9: Publication Table

In [ ]:
from clean import create_publication_table

# Create publication-ready table
pub_table = create_publication_table(
    list(robust_results.values()),
    model_names=['Baseline', 'No Outliers', 'Winsorized', 'Pre-2008', 'Post-2008'],
    output_format='text'
)

print(pub_table)

## 📝 Part 10: Generate Methods Section

In [ ]:
from clean import generate_methods_section

# Auto-generate data section for paper
methods = generate_methods_section(
    master,
    output_path=REPO_ROOT / "paper" / "methods.md"
)

print("Methods section generated! Preview:")
print(methods[:500] + "...")

## 🔄 Part 11: Export to R/Stata

In [ ]:
from clean.export import export_to_r, export_to_stata_script

# Generate R script with pre-configured models
export_to_r(
    master_with_lags,
    REPO_ROOT / 'analysis' / 'analysis.R',
    include_packages=['fixest', 'plm', 'did']
)

# Generate Stata script
export_to_stata_script(
    master_with_lags,
    REPO_ROOT / 'analysis' / 'analysis.do'
)

## ✅ Done!

**You now have:**
- ✅ Processed and merged datasets
- ✅ Quality reports
- ✅ Statistical tests
- ✅ Lags for dynamic models
- ✅ Robustness checks
- ✅ Publication tables
- ✅ Auto-generated methods section
- ✅ R/Stata scripts

**Ready for research!** 🚀